# 03 · Advanced model: YOLOv8s + Indian roads + tuned training

Changes vs baseline (one reason each, for the slides):

| Change | Why |
|---|---|
| YOLOv8**s** instead of **n** | more capacity for varied road textures |
| + RDD2022 India potholes and background images | Indian roads, lighting and cameras; fewer false alarms on cracks/shadows |
| AdamW, lr0 0.001, cosine LR, 120 epochs | smoother convergence on a bigger dataset |
| Road-specific augmentation (no vertical flip, strong brightness/saturation) | dashcam lighting varies; roads are never upside down |

Config: `configs/advanced.yaml`. Run on a GPU (Colab T4: ~1-1.5 h).

In [ ]:
# --- Setup: works locally and on Google Colab (free T4 GPU) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Upload the project folder to Google Drive first (MyDrive/Road-Damage-Pothole-Detection)
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/Road-Damage-Pothole-Detection")
    !pip -q install -r requirements.txt
else:
    # run from the notebooks/ folder -> go to the project root
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Project root:", os.getcwd())

In [ ]:
!python -m pothole.train --config configs/advanced.yaml

## Optional: hyperparameter experiments
Each run is logged to `reports/experiments.csv`. Change one thing at a time.

In [ ]:
# !python -m pothole.train --config configs/advanced.yaml --name adv_img800 --imgsz 800
# !python -m pothole.train --config configs/advanced.yaml --name adv_yolo11s --model yolo11s.pt
# !python -m pothole.train --config configs/advanced.yaml --name adv_no_mixup --mixup 0.0

In [ ]:
import pandas as pd
pd.read_csv('reports/experiments.csv')[['name','model','epochs','imgsz','optimizer','precision','recall','mAP50','mAP50_95','train_minutes']]

## Optimisation: export for deployment

In [ ]:
!python -m pothole.export --model models/advanced_yolov8s_india.pt --formats onnx
# Android / edge: !python -m pothole.export --model models/advanced_yolov8s_india.pt --formats tflite --int8 --data data/processed/merged/data.yaml